# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q Faker statsmodels pandas numpy matplotlib seaborn
    print('Packages installed.')


# Conjoint Analysis (Rating-Based) In-Depth
This notebook demonstrates how to perform a Rating-Based Conjoint Analysis. We will generate synthetic data representing consumer preferences for a product (e.g., a Smartphone) with varying attributes (Price, Brand, Storage) and estimate the utility of each attribute level using OLS regression.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
import itertools

fake = Faker()
np.random.seed(42)
sns.set_style("whitegrid")

## 1. Define Attributes and Levels
We are designing a study for a new **Smartphone**. The attributes are:
*   **Brand**: Apple, Samsung, Google
*   **Price**: $500, $800, $1100
*   **Storage**: 128GB, 256GB, 512GB

We will generate a **Full Factorial Design** (all possible combinations).

In [ ]:
attributes = {
    'Brand': ['Apple', 'Samsung', 'Google'],
    'Price': ['$500', '$800', '$1100'],
    'Storage': ['128GB', '256GB', '512GB']
}

keys = list(attributes.keys())
levels = list(attributes.values())

# Generate all combinations
profiles = list(itertools.product(*levels))
df_profiles = pd.DataFrame(profiles, columns=keys)
print(f"Total unique profiles: {len(df_profiles)}")
df_profiles.head()

## 2. Generate Synthetic Survey Data
We simulate 100 respondents. Each respondent rates all 27 profiles on a scale of 0 to 10.
We assume specific preferences to make the results interpretable:
*   **Price**: Lower is better (negative utility).
*   **Storage**: Higher is better (positive utility).
*   **Brand**: Apple > Samsung > Google (just for this simulation).

In [ ]:
data = []

for _ in range(100): # 100 Respondents
    respondent_id = fake.uuid4()
    
    for idx, row in df_profiles.iterrows():
        # Base utility
        utility = 5
        
        # Brand utility simulation
        if row['Brand'] == 'Apple': utility += 2
        elif row['Brand'] == 'Samsung': utility += 1
        
        # Price utility simulation (Lower is better)
        if row['Price'] == '$500': utility += 3
        elif row['Price'] == '$800': utility += 1
        elif row['Price'] == '$1100': utility -= 2
        
        # Storage utility simulation
        if row['Storage'] == '512GB': utility += 2
        elif row['Storage'] == '256GB': utility += 1
        
        # Add noise
        rating = utility + np.random.normal(0, 1.5)
        rating = max(0, min(10, rating)) # Clip to 0-10
        
        data.append([respondent_id] + row.tolist() + [rating])

df = pd.DataFrame(data, columns=['RespondentID'] + keys + ['Rating'])
print(f"Total observations: {len(df)}")
df.head()

## 3. Prepare Data for Regression
Conjoint analysis typically uses **Dummy Variables** to represent levels. We use `pd.get_dummies`.
Note: We must drop one level per attribute to avoid multicollinearity (the Reference Level).

In [ ]:
X = df[keys]
y = df['Rating']

# Convert to dummies
X_dummy = pd.get_dummies(X, drop_first=True)
X_dummy = sm.add_constant(X_dummy)

print("Features (Reference levels dropped):")
print(X_dummy.columns.tolist())

## 4. Estimate Part-Worth Utilities (OLS)
We run a linear regression. The coefficients represent the "Part-Worth" (utility) of each level relative to the reference level.

In [ ]:
model = sm.OLS(y, X_dummy.astype(float)).fit()
print(model.summary())

## 5. Calculate and Visualize Importance
The range of utility values for an attribute (Max - Min) determines its importance.

In [ ]:
# Extract coefficients
coefs = model.params.to_dict()

# Reconstruct full utilities including reference levels (which have utility 0 relative to themselves)
# We need to manually map back. 

utilities = {}

for attr, lvls in attributes.items():
    utilities[attr] = {}
    for lvl in lvls:
        dummy_col = f"{attr}_{lvl}"
        if dummy_col in coefs:
            utilities[attr][lvl] = coefs[dummy_col]
        else:
            # This is the reference level (dropped column)
            utilities[attr][lvl] = 0

# Calculate Importance
importances = {}
total_range = 0

for attr, values in utilities.items():
    u_vals = list(values.values())
    attr_range = max(u_vals) - min(u_vals)
    importances[attr] = attr_range
    total_range += attr_range

# Normalize to percentage
relative_importance = {k: (v / total_range) * 100 for k, v in importances.items()}

# Visualize
importance_df = pd.DataFrame(list(relative_importance.items()), columns=['Attribute', 'Importance'])
importance_df = importance_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x='Importance', y='Attribute', palette='viridis')
plt.title('Relative Importance of Attributes')
plt.xlabel('Importance (%)')
plt.show()

In [ ]:
print("Part-Worth Utilities:")
for attr, vals in utilities.items():
    print(f"\n{attr}:")
    for lvl, util in vals.items():
        print(f"  {lvl}: {util:.4f}")